# Road-Safety & Traffic-Law Assistant: SFT with QLoRA
Base model **Qwen2.5-1.5B-Instruct**, 4-bit NF4 quantization, LoRA (r=16), trained on a free Colab **T4 GPU**.

Setup: Runtime -> Change runtime type -> T4 GPU. Add a Colab secret named `HF_TOKEN` (write access) only if you want to upload the adapter in step 8.

## 1. Install libraries

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate gradio

## 2. Get the data
Loads `road_safety.jsonl` (train) and `heldout_test.jsonl` (test) from the project repo. If the clone fails, upload the two files manually.

In [ ]:
import os, shutil, json
REPO = "https://github.com/lisaluis/road-safety-practice"
if not (os.path.exists("road_safety.jsonl") and os.path.exists("heldout_test.jsonl")):
    !git clone -q {REPO} repo
    for f in ["road_safety.jsonl", "heldout_test.jsonl"]:
        p = f"repo/llm/{f}"
        if os.path.exists(p):
            shutil.copy(p, f)
if not (os.path.exists("road_safety.jsonl") and os.path.exists("heldout_test.jsonl")):
    from google.colab import files
    print("Upload road_safety.jsonl and heldout_test.jsonl")
    files.upload()

train_rows = [json.loads(l) for l in open("road_safety.jsonl") if l.strip()]
held = [json.loads(l)["messages"] for l in open("heldout_test.jsonl") if l.strip()]
print(len(train_rows), "train examples,", len(held), "held-out examples")
print(train_rows[0])

## 3. Load the base model in 4-bit and test it BEFORE training

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb, device_map="auto", dtype=torch.float16)

questions = [m[0]["content"] for m in held[:5]]

def ask(m, q):
    text = tok.apply_chat_template([{"role": "user", "content": q}],
                                   tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=120, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("===== BEFORE TRAINING =====")
for q in questions:
    print("\nQ:", q, "\nA:", ask(model, q))

## 4. Fine-tune with QLoRA (SFT)
LoRA r=16, alpha=32 on all attention and MLP projections. 4 epochs, learning rate 2e-4, effective batch 8.

In [ ]:
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

ds = load_dataset("json", data_files="road_safety.jsonl", split="train")
print(len(ds), "examples")

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
                  task_type="CAUSAL_LM")
cfg = SFTConfig(output_dir="out", num_train_epochs=4,
                per_device_train_batch_size=2, gradient_accumulation_steps=4,
                learning_rate=2e-4, logging_steps=5, fp16=True, bf16=False,
                max_length=512, save_strategy="no", report_to="none")

trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds,
                     peft_config=lora, processing_class=tok)
for p in trainer.model.parameters():      # keep trainable LoRA weights in fp32 (T4 fp16 fix)
    if p.requires_grad:
        p.data = p.data.float()
trainer.train()

## 5. Test AFTER training (same held-out questions)

In [ ]:
trainer.model.eval()
print("===== AFTER TRAINING =====")
for q in questions:
    print("\nQ:", q, "\nA:", ask(trainer.model, q))

## 6. Extra checks: facts in training, and an out-of-scope question (should say it is not sure)

In [ ]:
extra = [
    "What is the fine for driving without a helmet?",
    "What is the penalty for driving without a licence?",
    "What is the fine for parking in a no-parking zone in Pune?",
]
for q in extra:
    print("\nQ:", q, "\nA:", ask(trainer.model, q))

## 7. Compute and memory footprint

In [ ]:
!nvidia-smi

## 8. Save the LoRA adapter and upload to Hugging Face
Needs a Colab secret `HF_TOKEN` with write access. Change `REPO_ID` to your own username.

In [ ]:
from huggingface_hub import HfApi, create_repo
from google.colab import userdata

REPO_ID = "lisaluis717/road-safety-lora-1p5b"
trainer.model.save_pretrained("road-safety-lora-1p5b")
tok.save_pretrained("road-safety-lora-1p5b")

token = userdata.get("HF_TOKEN")          # never print the token
create_repo(REPO_ID, token=token, exist_ok=True)
HfApi(token=token).upload_folder(folder_path="road-safety-lora-1p5b", repo_id=REPO_ID)
print("https://huggingface.co/" + REPO_ID)

## 9. Chat interface (Gradio)
Run this last: the cell stays running while the UI is up. Screenshot the UI with the public link visible.

In [ ]:
import gradio as gr

def chat(message, history):
    return ask(trainer.model, message)

gr.ChatInterface(
    chat,
    title="Road Safety & Traffic Law Assistant",
    description="Qwen2.5-1.5B-Instruct fine-tuned with QLoRA (4-bit) on Indian traffic-law Q&A.",
    examples=["What is the fine for driving without a helmet?",
              "How have road deaths in India changed since 2015?",
              "What is the fine for parking in a no-parking zone in Pune?"],
).launch(share=True, debug=True)